# 🧪 GraphRAG test suite

27 questions written against the seed corpus in Section B.3b of your notebook, so every one is
answerable and each tests something specific.

**How to use this file:** copy the code cell below into your notebook (paste it as a new cell
at the very end), then run the two cells after it. Nothing here connects to anything on its
own — it only defines questions and runners that call *your* `ask()`, `route()` and `neighbors()`.

Run in this order. The first two cost almost nothing, and if either fails, everything below it
fails for reasons that have nothing to do with retrieval.

| Call | Cost | What it tells you |
|---|---|---|
| `inspect_the_graph()` | free | Is the graph built? Can you reach the key chains? |
| `test_routing()` | ~1 call/question | Do questions reach the right engine? |
| `test_graph_beats_rag()` | moderate | Does the graph find anything plain RAG doesn't? |
| `test_traps()` | moderate | Does it stay careful when two figures look alike? |
| `test_refusals()` | moderate | Does it refuse, or invent? |
| `test_global()` | moderate | Needs Section H to have been built |
| `run_all()` | all of it | Everything, in order |

**The single most useful diagnostic:** if `neighbors("CERT-TW-SQF2-STK", hops=2)` does not show
`CTR-2024-0489-TFH`, the flagship multi-hop question *cannot* work — and the problem is the
graph BUILD (Sections G.3–G.5), not retrieval, not the prompt, not the model.

---

### The three questions worth trying first

1. *"What is the relationship between Tidewater Frozen Holdings and SUP-1157?"*
   Should route to 🕸️ GRAPH and cite `SUBSIDIARY_OF` as `[graph]`. If it doesn't, fix the build.
2. *"When does Tidewater's Stockton SQF certificate expire, and what happens if it lapses?"*
   The date is in the certificate register; the consequence is a contract clause that never says
   "SQF" or "Tidewater". Plain RAG finds the date and misses the consequence.
3. *"Why does the Tidewater contract depend on a company that is not a party to it?"*
   Three hops, no single document states it. The hardest question in the corpus.

In [ ]:
# ============================================================================
#  GraphRAG test suite — written against YOUR seed corpus (Section B.3b)
#
#  Paste as a new cell at the very END of the notebook, after Section H.
#  Then:
#      test_routing()        # cheap: checks WHERE each question goes. ~1 call each.
#      test_graph_beats_rag()# the money shot: same question, graph vs plain RAG
#      test_traps()          # the questions that catch hallucination
#      test_global()         # needs build_communities() + summarize_communities()
#      run_all()             # everything, in order
#
#  Every question below is answerable from the 16 seeded documents. The entities
#  are real: SUP-1042 Valle Verde Produce, SUP-1077 Acme Foods, SUP-1103 Globex
#  Dairy, SUP-1156 Tidewater Frozen Holdings, SUP-1189 Northlake Protein,
#  SUP-1157 (Tidewater's subsidiary), and 137 relationship triples between them.
# ============================================================================

# ---------------------------------------------------------------------------
#  A. THE CHAINS YOUR GRAPH ACTUALLY CONTAINS
#     Read these first — they tell you WHY each question below is a fair test.
#
#   1. CERTIFICATE → CONTRACT
#        CERT-TW-SQF2-STK  --COVERS_FACILITY-->  FAC-TW-STOCKTON
#        CERT-TW-SQF2-STK  --REQUIRED_BY----->   CTR-2024-0489-TFH
#        SUP-1156          --OPERATES_FACILITY-> FAC-TW-STOCKTON
#        SUP-1156          --PARTY_TO--------->  CTR-2024-0489-TFH
#      A certificate's expiry date and the contract clause that depends on it
#      live in DIFFERENT documents that share almost no vocabulary.
#
#   2. THE SUBSIDIARY CHAIN  ← the best test in the whole corpus
#        SUP-1157  --SUBSIDIARY_OF-->  SUP-1156      (Tidewater owns it)
#        SUP-1156  --ACQUIRED------->  SUP-1157
#        SUP-1156  --LIABLE_FOR----->  SUP-1157
#        SUP-1157  --HOLDS_CERTIFICATION--> CERT-HP-MSC
#        CERT-HP-MSC --REQUIRED_BY--> CTR-2024-0489-TFH
#      A contract signed by the PARENT depends on a certificate held by the
#      SUBSIDIARY. Three hops. No single document states this.
#
#   3. THE COMPETITIVE CHAIN
#        BID-2024-089 (Valle Verde) SCORED 91.4, AWARDED, → CTR-2024-0412-VV
#        BID-2024-091 (Acme)        SCORED 74.7, LOST_TO BID-2024-089
#
#   4. THE PERFORMANCE CHAIN
#        ORG-RUSD --REQUESTED_CORRECTIVE_ACTION_FROM--> SUP-1156
#        SUP-1156 --SATISFACTION_RATING--> 4.1/5   (the lowest of five)
#        SUP-1156 --DECLINED_TO_BID--> RFP-2024-0412 and RFP-2024-0455
# ---------------------------------------------------------------------------


TESTS = [
    # ======================= SQL — computed from columns =======================
    dict(q="How many RFPs are there in total?",
         expect="sql",
         why="A count over rows. Nothing in a document states this number.",
         look_for="a number, returned as a DataFrame"),

    dict(q="Which supplier has the highest total bid amount, and in which category?",
         expect="sql",
         why="Ranking + aggregation. The classic SQL shape.",
         look_for="a table, not prose"),

    dict(q="List all bids submitted after March 2024, newest first.",
         expect="sql",
         why="A filtered, ordered list of records.",
         look_for="rows with dates in order"),

    # ======================= RAG — one fact, one document =====================
    dict(q="What are Globex Dairy Cooperative's payment terms?",
         expect="rag",
         why="A single fact stated in prose in one document.",
         look_for="the terms, with a [n] citation"),

    dict(q="What does bid BID-2024-089 quote for product VV-PRD-1180?",
         expect="rag",
         why="Exact-token lookup — this is where the KEYWORD leg earns its place. "
             "Pure vector search struggles with codes like VV-PRD-1180.",
         look_for="a price or spec, cited"),

    dict(q="What is the delivery window specified in RFP-2024-0412?",
         expect="rag",
         why="One clause, one document.",
         look_for="a window, cited"),

    # ======================= GRAPH — the multi-hop questions ==================
    dict(q="When does Tidewater's Stockton SQF certificate expire, and what happens "
           "if it lapses?",
         expect="graph",
         why="CHAIN 1. The expiry date is in the certificate register; the consequence "
             "is a clause in CTR-2024-0489-TFH that never mentions 'SQF' or 'Tidewater'. "
             "Plain RAG finds the date and misses the consequence.",
         look_for="BOTH the expiry date AND the contractual consequence"),

    dict(q="Why does the Tidewater contract depend on a company that is not a party to it?",
         expect="graph",
         why="CHAIN 2 — the hardest question in the corpus. Requires: contract requires "
             "CERT-HP-MSC → held by SUP-1157 → subsidiary of SUP-1156 → which is the "
             "contracting party. Three hops, no single document.",
         look_for="the subsidiary relationship AND the certificate dependency"),

    dict(q="What is the relationship between Tidewater Frozen Holdings and SUP-1157?",
         expect="graph",
         why="Names two entities and asks how they relate — the textbook graph shape.",
         look_for="SUBSIDIARY_OF / ACQUIRED / LIABLE_FOR, cited as [graph]"),

    dict(q="What happens to the Valleverde contract if their cold-chain certification lapses?",
         expect="graph",
         why="Consequence question: CERT-VV-SQF2 --REQUIRED_BY--> CTR-2024-0412-VV.",
         look_for="the contractual consequence, not just the certificate"),

    dict(q="Which facilities would be affected if Tidewater lost its Stockton certification, "
           "and which contracts depend on those facilities?",
         expect="graph",
         why="Two hops outward, then back into contracts. Tests that expansion is "
             "reaching hop 2 and not stopping at hop 1.",
         look_for="FAC-TW-STOCKTON and CTR-2024-0489-TFH both named"),

    dict(q="Everything we know about Tidewater Frozen Holdings.",
         expect="graph",
         why="'Everything about X' where X appears across bids, contracts, certificates, "
             "facilities and a corrective-action notice. Tests breadth of expansion.",
         look_for="facts from at least three different document types"),

    dict(q="Which supplier lost the produce RFP, and what did the winner score?",
         expect="graph",
         why="CHAIN 3. LOST_TO connects two bids that are described in separate documents.",
         look_for="Acme (74.7) lost to Valle Verde (91.4)"),

    dict(q="Why did Riverside request corrective action, and did that supplier bid on "
           "anything afterwards?",
         expect="graph",
         why="CHAIN 4. Corrective action, the rating, and two DECLINED_TO_BID edges — "
             "three documents, one causal story.",
         look_for="the corrective action AND the declined bids"),

    dict(q="Can Central Valley USD buy under Riverside's produce contract, and what "
           "conditions apply?",
         expect="graph",
         why="MAY_PIGGYBACK_ON — a relationship stated in one document with conditions "
             "stated in another.",
         look_for="the piggyback right AND its conditions"),

    # ======================= GLOBAL — needs Section H =========================
    dict(q="What are the recurring compliance and delivery risks across all suppliers?",
         expect="global",
         why="No chunk contains this. It only exists as a pattern across the corpus.",
         look_for="themes spanning several suppliers, not one supplier's detail"),

    dict(q="What are the main themes in how suppliers respond to our RFPs?",
         expect="global",
         why="'Main themes' + 'suppliers' plural — corpus-wide by construction.",
         look_for="patterns, each attributed to named suppliers"),

    dict(q="Overall, what is this procurement programme most exposed to?",
         expect="global",
         why="A question about the whole corpus with no named entity at all.",
         look_for="risks synthesised across communities"),

    dict(q="Summarise the certification landscape across all our suppliers.",
         expect="global",
         why="Requires seeing every certificate at once, not retrieving one.",
         look_for="SQF, GAP, BRCGS, MSC and who holds what"),

    # ======================= TRAPS — where systems hallucinate ================
    dict(q="What is our supplier satisfaction rating?",
         expect=None,
         why="⚠️ THE TRAP. 4.8/5 is ONE supplier's rating (Valle Verde). There is also a "
             "district-wide index of 8.5/10. A careless system merges them or reports "
             "one as the other.",
         look_for="BOTH figures, each attributed — or a request to clarify. "
                  "A single blended number is a FAILURE."),

    dict(q="What certifications does Tidewater hold?",
         expect=None,
         why="⚠️ TRAP. Tidewater (SUP-1156) holds certs for its own facilities; CERT-HP-MSC "
             "belongs to its SUBSIDIARY SUP-1157. Attributing the subsidiary's certificate "
             "to the parent is wrong, and easy to do.",
         look_for="Tidewater's own certificates, with the subsidiary's noted SEPARATELY"),

    dict(q="What was Acme's score on the produce RFP?",
         expect=None,
         why="⚠️ TRAP. 74.7 belongs to BID-2024-091. 91.4 belongs to Valle Verde. Systems "
             "that retrieve both documents often return the wrong number.",
         look_for="74.7, attributed to Acme — not 91.4"),

    dict(q="Which supplier operates the Oxnard facility?",
         expect=None,
         why="⚠️ TRAP. FAC-HP-OXNARD is operated by SUP-1157, the subsidiary — not by "
             "Tidewater, even though Tidewater owns SUP-1157.",
         look_for="SUP-1157, with the ownership noted"),

    # ======================= REFUSALS — should NOT invent =====================
    dict(q="What is the capital of France?",
         expect="none",
         why="General knowledge. Must be refused, not answered.",
         look_for="a refusal and a hint about what CAN be asked"),

    dict(q="What were our supplier ratings in 2019?",
         expect=None,
         why="Plausible, on-topic, and absent from the corpus. The hardest refusal.",
         look_for='exactly the REFUSAL string: "I don\'t have that in the indexed documents."'),

    dict(q="Which supplier has the best cybersecurity posture?",
         expect=None,
         why="On-topic vocabulary, no such data anywhere. Tests the relevance floor.",
         look_for="a refusal, NOT an inference from unrelated documents"),

    dict(q="Hello, how are you?",
         expect="none",
         why="Chit-chat. Should be refused cleanly by the router.",
         look_for="the 'ask something about your tables or documents' message"),
]


# ---------------------------------------------------------------------------
#  RUNNERS
# ---------------------------------------------------------------------------
def _actual_engine(q):
    """Where would ask() send this? Mirrors ask()'s own logic, without answering."""
    try:
        if _is_global(q, verbose=False):          # Section H, if installed
            return "global"
    except NameError:
        pass
    try:
        return route(q)["engine"]
    except Exception as e:
        return f"ERROR:{type(e).__name__}"


def test_routing(only=None):
    """Cheap. Checks WHERE each question goes, without paying to answer it.

    Run this FIRST. If routing is wrong, every answer below it is wrong for a
    reason that has nothing to do with retrieval.
    """
    rows, wrong = [], 0
    tests = [t for t in TESTS if t["expect"]] if only is None else \
            [t for t in TESTS if t["expect"] == only]
    print(f"Routing {len(tests)} question(s)…\n")
    for t in tests:
        got = _actual_engine(t["q"])
        ok = (got == t["expect"])
        wrong += (not ok)
        rows.append({"expected": t["expect"], "got": got, "ok": "✅" if ok else "❌",
                     "question": t["q"][:66]})
    df = pd.DataFrame(rows)
    display(df)
    print(f"\n{len(tests) - wrong}/{len(tests)} routed as expected.")
    if wrong:
        print("A wrong route is usually one of three things:")
        print("  • the router crashed and failed open to RAG — look for ⚠️ ROUTER FAILED")
        print("  • the graph is not built, so route() rewrites 'graph' to 'rag'")
        print("  • no community summaries exist, so global questions fall back")
    return df


def test_graph_beats_rag():
    """The comparison that proves the graph is doing something.

    Same question, both engines. If the answers are identical, the graph is not
    earning its keep FOR THAT QUESTION — which is real information, not a bug.
    """
    pairs = [t for t in TESTS if t["expect"] == "graph"][:4]
    for t in pairs:
        print("\n" + "=" * 78)
        print("Q:", t["q"])
        print("   look for:", t["look_for"])
        print("=" * 78)
        print("\n--- 🕸️  GRAPH ---")
        g = ask(t["q"], mode="graph", verbose=False)
        print("\n--- 📄 PLAIN RAG ---")
        show(ask_rag(t["q"]))
        print("\n💡 Did the graph answer contain something RAG's did not?")
        print("   Look for sources tagged '← pulled in by the graph'. If none ever")
        print("   appear, expansion is contributing nothing.")


def test_traps():
    """The questions that separate a careful system from a confident one."""
    for t in [x for x in TESTS if "TRAP" in x["why"]]:
        print("\n" + "=" * 78)
        print("Q:", t["q"])
        print("⚠️ ", " ".join(t["why"].split())[:200])
        print("   PASS =", t["look_for"])
        print("=" * 78)
        ask(t["q"])


def test_refusals():
    """A system that never refuses is not safe — it is just confident."""
    for t in [x for x in TESTS if "refus" in x["look_for"].lower()
              or "Refus" in x["why"] or "absent" in x["why"]]:
        print("\n" + "-" * 78)
        print("Q:", t["q"], "\n   PASS =", t["look_for"])
        ask(t["q"])


def test_global():
    """Needs build_communities() + summarize_communities() to have been run."""
    try:
        with db(dict_rows=True) as cur:
            cur.execute(f"SELECT count(*) AS n FROM {COMMUNITY_TABLE} "
                        f"WHERE summary IS NOT NULL")
            n = cur.fetchone()["n"]
    except Exception:
        print("Section H is not installed — no global search in this notebook.")
        return
    if not n:
        print("No community summaries yet. Run build_communities() then "
              "summarize_communities() first.")
        return
    for t in [x for x in TESTS if x["expect"] == "global"]:
        print("\n" + "=" * 78)
        print("Q:", t["q"], "\n   look for:", t["look_for"])
        print("=" * 78)
        ask(t["q"], mode="global")


def inspect_the_graph():
    """Before blaming retrieval, look at what the graph actually knows."""
    print("── Is the graph even built? ──")
    with db(dict_rows=True) as cur:
        for label, t in [("nodes", NODE_TABLE), ("edges", EDGE_TABLE),
                         ("mentions", MENTION_TABLE)]:
            cur.execute(f"SELECT count(*) AS n FROM {t}")
            print(f"   {label:<10} {cur.fetchone()['n']:>7,}")
        cur.execute(f"""SELECT predicate, count(*) AS n FROM {EDGE_TABLE}
                        GROUP BY predicate ORDER BY n DESC LIMIT 12""")
        print("   top predicates:",
              ", ".join(f"{r['predicate']}={r['n']}" for r in cur.fetchall()))
        cur.execute(f"""SELECT method, count(*) AS n FROM {MENTION_TABLE}
                        GROUP BY method ORDER BY n DESC""")
        print("   mentions by method:",
              ", ".join(f"{r['method']}={r['n']}" for r in cur.fetchall()))
        print("   ↑ mostly 'label' means entity linking is guesswork — expect noise.")

    print("\n── Can you reach the key chains? ──")
    print("find_node('Tidewater'):");            find_node("Tidewater")
    print("\nneighbors('SUP-1156', hops=2):");   neighbors("SUP-1156", hops=2)
    print("\nneighbors('CERT-TW-SQF2-STK', hops=2):")
    neighbors("CERT-TW-SQF2-STK", hops=2)
    print("\n↑ If CTR-2024-0489-TFH does NOT appear within 2 hops of that certificate,")
    print("  the flagship question CANNOT work, and the problem is the BUILD")
    print("  (Sections G.3–G.5), not retrieval or the model.")


def run_all():
    print("=" * 78); print("1 · IS THE GRAPH BUILT?"); print("=" * 78)
    inspect_the_graph()
    print("\n" + "=" * 78); print("2 · ROUTING (cheap)"); print("=" * 78)
    test_routing()
    print("\n" + "=" * 78); print("3 · GRAPH vs PLAIN RAG"); print("=" * 78)
    test_graph_beats_rag()
    print("\n" + "=" * 78); print("4 · TRAPS"); print("=" * 78)
    test_traps()
    print("\n" + "=" * 78); print("5 · REFUSALS"); print("=" * 78)
    test_refusals()
    print("\n" + "=" * 78); print("6 · GLOBAL"); print("=" * 78)
    test_global()


print(f"✅ {len(TESTS)} test questions loaded.")
print("   inspect_the_graph()    ← start here, costs nothing")
print("   test_routing()         ← then this")
print("   test_graph_beats_rag() · test_traps() · test_refusals() · test_global()")
print("   run_all()")


In [ ]:
inspect_the_graph()   # free — start here

In [ ]:
test_routing()        # cheap — then this

---

Then, when routing looks right:

```python
test_graph_beats_rag()
test_traps()
test_refusals()
test_global()
```

or `run_all()` for the lot.